# Company Data Quality
Exploratory analysis of `out/04_normalized.json` after the full pipeline

In [18]:
import json
from collections import defaultdict
from pathlib import Path

data = json.loads(Path("out/04_normalized.json").read_text())
total = len(data)
print(f"Total companies: {total}")

Total companies: 3082


## Completeness

In [19]:
def pct(n): return f"{n}/{total} ({n/total*100:.1f}%)"

def has_contact(c, t): return any(x["type"] == t for x in c.get("companyContacts", []))

fields = {
    "Has email":    sum(1 for c in data if has_contact(c, "email")),
    "Has tel":      sum(1 for c in data if has_contact(c, "tel")),
    "Has fax":      sum(1 for c in data if has_contact(c, "fax")),
    "Has website":  sum(1 for c in data if has_contact(c, "website")),
    "Has taxId":    sum(1 for c in data if c.get("taxId")),
    "Has address":  sum(1 for c in data if c.get("addresses")),
    "Has region":   sum(1 for c in data if c.get("region")),
    "Has country":  sum(1 for c in data if c.get("country")),
    "Has nameCn":   sum(1 for c in data if c.get("companyNameZh")),
    "Has nameVi":   sum(1 for c in data if c.get("companyNameVi")),
    "Has nameEn":   sum(1 for c in data if c.get("companyNameEn")),
    "Has userContact": sum(1 for c in data if c.get("userContacts")),
}

for label, count in fields.items():
    print(f"  {label:<18} {pct(count)}")

  Has email          2966/3082 (96.2%)
  Has tel            2778/3082 (90.1%)
  Has fax            849/3082 (27.5%)
  Has website        1401/3082 (45.5%)
  Has taxId          2938/3082 (95.3%)
  Has address        3082/3082 (100.0%)
  Has region         3082/3082 (100.0%)
  Has country        3061/3082 (99.3%)
  Has nameCn         2847/3082 (92.4%)
  Has nameVi         2502/3082 (81.2%)
  Has nameEn         790/3082 (25.6%)
  Has userContact    3040/3082 (98.6%)


## Country Distribution

In [20]:
counts = defaultdict(int)
for c in data:
    counts[c.get("country") or "(none)"] += 1
for k, v in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<6} {v:>5}  {v/total*100:.1f}%")

  TW      2316  75.1%
  CN       419  13.6%
  VN       182  5.9%
  HK        59  1.9%
  SG        39  1.3%
  JP        26  0.8%
  (none)    21  0.7%
  MY         7  0.2%
  TH         4  0.1%
  KR         2  0.1%
  SE         1  0.0%
  SA         1  0.0%
  AU         1  0.0%
  FR         1  0.0%
  CH         1  0.0%
  US         1  0.0%
  NL         1  0.0%


## Region Distribution

In [21]:
counts = defaultdict(int)
for c in data:
    counts[c.get("region") or "(none)"] += 1
for k, v in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {v:>5}  {k}")

   1972  Ho Chi Minh City
    402  Dong Nai
    306  Other
    222  Tay Ninh
    139  Ha Noi
     41  Lam Dong


## Industry Distribution

In [22]:
counts = defaultdict(int)
for c in data:
    for ind in c.get("industries", []):
        counts[ind] += 1
for k, v in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {v:>5}  {k}")

    376  metal
    357  plastic
    316  construction
    298  textile
    273  machinery
    225  paper
    218  furniture
    204  electronics
    200  shoes
    182  other
    180  vehicle
    118  agriculture
    108  food
     98  logistics
     85  tourism
     85  gifts
     70  legal
     48  finance
     41  education


## Companies Without Email
These 118 companies will create a Company record only - no User account.

In [23]:
no_email = [c for c in data if not has_contact(c, "email")]
print(f"No email: {len(no_email)}\n")
for c in no_email[:20]:
    name = c.get("companyNameEn") or c.get("companyNameVi") or c.get("companyNameZh") or "?"
    print(f"  [{c.get('country','?')}] {name}")
print(f"...")

No email: 116

  [TW] JUNMAY LABEL CO.,LTD
  [TW] NING AN COMPANY LIMITED
  [SG] STARKWELL TECHNOLOGY PTE.LTD - VPDD TAI HA NOI
  [TW] NOVA CO.,LTD
  [TW] TAY MING CO.,LTD
  [TW] SMITH MFG VIETNAM CO.,LTD
  [TW] ASIA PACIFIC PLASTIC CO.,LTD
  [SG] APL – NOL VIETNAM LIMITED
  [CN] AGRICULTURAL BANK OF CHINA LIMIED - HANOI BRANCH
  [None] BO BIT TET COM PHAN LAU
  [VN] YEEBO SEAFOOD & HOT POT RESTAURANT
  [TW] CUA HANG LANG TRA ALISHAN
  [TW] ROYAL ISLAND GOLF & VILLAS
  [TW] SUI CAO DAI NUONG
  [TW] KIKI HOTPOT
  [TW] MONICA HAIR SALON & SPA
  [TW] CONG TY TNHH MAY MAC NGHIA TUNG
  [TW] CONG TY TNHH APPAREL FAR EASTERN
  [TW] CONG TY TNHH HOA CHAT HSIN SOU VN
  [CN] CONG TY TNHH HISHENG LUGGAGE AND GARMENT ACCESSORY
...


## Companies Without Tax ID

In [24]:
no_taxid = [c for c in data if not c.get("taxId")]
print(f"No taxId: {len(no_taxid)}\n")
for c in no_taxid[:5]:
    name = c.get("companyNameEn") or c.get("companyNameVi") or c.get("companyNameZh") or "?"
    print(f"  [{c.get('country','?')}] {name}")
print(f"...")

No taxId: 144

  [CN] CHINA TEXMATECH CO.,LTD
  [CN] SAB WEIXING CO.,LTD
  [None] ANH GOING CO.,LTD
  [TW] TAIWAN TEXTILE FEDERATION
  [TW] PRECIOUS MOUNTAIN ENT. CORP.
...


## Multi-Industry Companies

In [25]:
multi = [c for c in data if len(c.get("industries", [])) > 1]
print(f"Multi-industry: {len(multi)}\n")
for c in multi[:5]:
    name = c.get("companyNameEn") or c.get("companyNameVi") or c.get("companyNameZh") or "?"
    print(f"  {c['industries']}  →  {name}")
print(f"...")

Multi-industry: 377

  ['textile', 'tourism']  →  ACODE SPORTING GOODS CO.,LTD
  ['textile', 'plastic']  →  CPH (VN) CO.,LTD
  ['textile', 'construction', 'other']  →  VAN BAO DUC TRADING CO.,LTD
  ['textile', 'shoes']  →  BETAMEX VIETNAM CO.,LTD
  ['textile', 'shoes']  →  PINE TEXTILE VN ONE MEMBER CO.,LTD
...


## Remaining Flags

In [26]:
flagged = [c for c in data if c.get("_flags")]
print(f"Flagged: {len(flagged)}\n")
for c in flagged:
    name = c.get("companyNameEn") or c.get("companyNameVi") or c.get("companyNameZh") or "?"
    print(f"  {name}")
    for f in c["_flags"]:
        print(f"    {repr(f)}")

Flagged: 0



## Companies With No Country

In [27]:
no_country = [c for c in data if not c.get("country")]
print(f"No country: {len(no_country)}\n")
for c in no_country:
    name = c.get("companyNameEn") or c.get("companyNameVi") or c.get("companyNameZh") or "?"
    raw = c.get("_raw") or {}
    raw_names = (raw.get("names", "") if isinstance(raw, dict) else "").replace("\n", " | ")
    print(f"  {name}")
    print(f"    raw: {raw_names[:80]}")

No country: 21

  ANH GOING CO.,LTD
    raw: 英弘貿易有限公司 | ANH GOING CO.,LTD
  TOP VENDING MACHINE ELECTRONICS CO.,LTD
    raw: 鼎鉅電子股份有限公司 | TOP VENDING MACHINE ELECTRONICS CO.,LTD
  STANDARD CHARTERED BANK (VIETNAM) LIMITED
    raw: 渣打銀行 | STANDARD CHARTERED BANK (VIETNAM) LIMITED
  BAO SAI GON GIAI PHONG HOA VAN
    raw: 西貢解放日報 | BAO SAI GON GIAI PHONG HOA VAN
  HARVEST SERVICE – TRADING - PRODUCTION COMPANY LIMITED
    raw: HARVEST SERVICE – TRADING - PRODUCTION COMPANY LIMITED | CONG TY TNHH SAN XUAT -
  BO BIT TET COM PHAN LAU
    raw: 七號火箭牛排館 | BO BIT TET COM PHAN LAU
  HOTEL MAJESTIC
    raw: HOTEL MAJESTIC
  WEST LAKES GOLF & VILLAS
    raw: WEST LAKES GOLF & VILLAS
  FLOWCOM VIET NAM COMPANY LIMITED
    raw: 越南弗絡肯有限公司 | FLOWCOM VIET NAM COMPANY LIMITED | --- | 越南弗絡肯有限公司 FLOWCOM VIET NAM 
  CONG TY TNHH NS BLUESCOPE VIET NAM
    raw: BLUESCOPE STEEL責任有限公司 | CONG TY TNHH NS BLUESCOPE VIET NAM
  CONG TY TNHH XAY DUNG VA CONG NGHE QT VIET NAM
    raw: 越南QT建築與科技責任有限公司 | CONG TY TNHH X

## Fraud Detection

### 1. Duplicate Emails
Same email registered to multiple companies - critical since email is the login identifier.

In [28]:
from collections import defaultdict

def get_emails(c):
    # deduplicate case-insensitively within a single company
    return list(dict.fromkeys(
        x["value"].lower().strip() for x in c.get("companyContacts", []) if x["type"] == "email"
    ))

def name(c):
    return c.get("companyNameEn") or c.get("companyNameVi") or c.get("companyNameZh") or "?"

email_map = defaultdict(list)
for c in data:
    for email in get_emails(c):
        email_map[email].append(c)

# Remove intra-company duplicates (same company added twice due to case variants)
def dedup_by_id(companies):
    seen = set()
    result = []
    for c in companies:
        key = (c.get("taxId"), c.get("companyNameZh"), c.get("companyNameVi"))
        if key not in seen:
            seen.add(key)
            result.append(c)
    return result

dup_emails = {e: dedup_by_id(cs) for e, cs in email_map.items()}
dup_emails = {e: cs for e, cs in dup_emails.items() if len(cs) > 1}
print(f"Duplicate emails: {len(dup_emails)}\n")
for email, companies in sorted(dup_emails.items(), key=lambda x: -len(x[1])):
    print(f"  {email}  ({len(companies)} companies)")
    for c in companies:
        print(f"    [{c.get('country','?')}] {name(c)} | taxId: {c.get('taxId','-')}")

Duplicate emails: 47

  sales1@everwin-group.com  (4 companies)
    [TW] CONG TY TNHH TU VAN VA KE TOAN EVERWIN | taxId: 0302776159
    [TW] CONG TY LUAT TNHH VINH NHAT | taxId: 0312379432
    [TW] CONG TY TNHH KIEM TOAN VA DINH GIA GAA | taxId: 0317916426
    [TW] CONG TY LUAT TNHH VINH NHAT | taxId: None
  leotsai@makalot.com.tw  (2 companies)
    [TW] TRIPLE GARMENT (VIETNAM) CO.,LTD | taxId: 0304968145
    [TW] CONG TY TNHH MAY MAC MAKALOT VIET NAM | taxId: 0800304871
  allen@xch-label.com  (2 companies)
    [TW] XIN CHANG HUA COMPANY LIMITED – BINH DUONG BRANCH | taxId: 3901106078-001
    [TW] XIN CHANG HUA LABEL WEAVING CO.,LTD | taxId: 3901106078
  yang.hsieh@chinli.com  (2 companies)
    [TW] CHIN LI PLASTIC INDUSTRIAL CO.,LTD | taxId: 3700230124
    [TW] CONG TY TNHH CHINLI MY PHUOC | taxId: 3702481613
  leolam@pouchen.com  (2 companies)
    [TW] POUYUEN VIETNAM CO.,LTD – HA NOI | taxId: 0300813662-003
    [TW] CONG TY TNHH POUYUEN VIET NAM | taxId: 0300813662
  sales@geogear.

### 2. Duplicate Tax IDs
Same MST assigned to different company names — either an alias or data error.

In [29]:
taxid_map = defaultdict(list)
for c in data:
    if c.get("taxId"):
        taxid_map[c["taxId"]].append(c)

dup_taxids = {t: cs for t, cs in taxid_map.items() if len(cs) > 1}
print(f"Duplicate tax IDs: {len(dup_taxids)}\n")
for taxid, companies in dup_taxids.items():
    print(f"  MST: {taxid}")
    for c in companies:
        emails = get_emails(c)
        print(f"    {name(c)} | email: {emails[:1] or '-'}")

Duplicate tax IDs: 0



### 3. Invalid Tax ID Format
Vietnamese MST must be 10 digits, or 13 digits for branches (`XXXXXXXXXX-XXX`).

In [30]:
import re

VN_TAXID_RE = re.compile(r"^\d{10}(-\d{3})?$")

invalid_taxid = [c for c in data if c.get("taxId") and not VN_TAXID_RE.match(c["taxId"])]
print(f"Invalid tax ID format: {len(invalid_taxid)}\n")
for c in invalid_taxid:
    print(f"  {repr(c['taxId']):<25}  {name(c)}")

Invalid tax ID format: 0



### 4. Phone Number Reuse
Same phone number shared across many companies.

In [31]:
def normalize_phone(p):
    return re.sub(r"[\s\-\.]", "", p)

phone_map = defaultdict(list)
for c in data:
    seen_phones = set()
    for contact in c.get("companyContacts", []):
        if contact["type"] in ("tel", "hotline"):
            norm = normalize_phone(contact["value"])
            if norm not in seen_phones:  # skip duplicates within same company
                seen_phones.add(norm)
                phone_map[norm].append(c)

dup_phones = {p: dedup_by_id(cs) for p, cs in phone_map.items()}
dup_phones = {p: cs for p, cs in dup_phones.items() if len(cs) > 2}  # threshold: >2
print(f"Phone numbers shared by more than 2 companies: {len(dup_phones)}\n")
for phone, companies in sorted(dup_phones.items(), key=lambda x: -len(x[1])):
    print(f"  {phone}  ({len(companies)} companies)")
    for c in companies[:5]:
        print(f"    {name(c)}")
    if len(companies) > 5:
        print(f"    ... and {len(companies)-5} more")

Phone numbers shared by more than 2 companies: 1

  02838603888  (7 companies)
    MEGA CHEMICAL CO.,LTD
    CONG TY TNHH TU VAN VA KE TOAN EVERWIN
    CONG TY LUAT TNHH VINH NHAT
    CONG TY TNHH TU VAN QUAN LY VA GIAI PHAP DAU TU EVERWIN
    CONG TY TNHH KIEM TOAN VA DINH GIA GAA
    ... and 2 more


## Type & Value Validation

In [32]:
import re

EXPECTED_FIELDS = {
    'companyNameZh', 'companyNameVi', 'companyNameEn', 'country', 'industry',
    'region', 'addresses', 'taxId', 'companyContacts', '_flags', 'description',
    'userContacts', '_raw', 'industries',
}

VALID_INDUSTRIES = {
    'agriculture', 'construction', 'education', 'electronics', 'finance',
    'food', 'furniture', 'gifts', 'legal', 'logistics', 'machinery', 'metal',
    'other', 'paper', 'plastic', 'shoes', 'textile', 'tourism', 'vehicle',
}

VALID_CONTACT_TYPES = {
    'tel', 'fax', 'email', 'website', 'hotline', 'skype', 'zalo', 'wechat', 'line', 'viber', 'facebook',
}

TAXID_RE  = re.compile(r'^\d{10}(-\d{3})?$')
EMAIL_RE  = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
ISO_RE    = re.compile(r'^[A-Z]{2}$')
WEBSITE_RE = re.compile(r'^https?://')

issues: dict[str, list[str]] = {}

def flag(c, msg):
    k = name(c)
    issues.setdefault(k, []).append(msg)

for c in data:
    n = name(c)

    # ── Unexpected top-level fields
    for k in c:
        if k not in EXPECTED_FIELDS:
            flag(c, f'unexpected field: {k!r}')

    # ── Scalar fields must be str or None (not list, int, etc.)
    for field in ('companyNameZh', 'companyNameVi', 'companyNameEn',
                  'country', 'taxId', 'description'):
        val = c.get(field)
        if val is not None and not isinstance(val, str):
            flag(c, f'{field} is {type(val).__name__}, expected str')
        if isinstance(val, str) and val.strip() == '':
            flag(c, f'{field} is empty string (should be null)')

    # ── List fields must be lists
    for field in ('addresses', 'companyContacts', 'userContacts', 'industries', '_flags'):
        val = c.get(field)
        if val is not None and not isinstance(val, list):
            flag(c, f'{field} is {type(val).__name__}, expected list')

    # ── region: should be str (not list after normalize)
    r = c.get('region')
    if isinstance(r, list):
        flag(c, f'region is still a list: {r}')

    # ── country: must be valid ISO alpha-2
    country = c.get('country')
    if country and not ISO_RE.match(country):
        flag(c, f'country not ISO: {country!r}')

    # ── taxId format
    tid = c.get('taxId')
    if tid and not TAXID_RE.match(tid):
        flag(c, f'invalid taxId: {tid!r}')

    # ── industries: only known values
    for ind in c.get('industries', []):
        if ind not in VALID_INDUSTRIES:
            flag(c, f'unknown industry: {ind!r}')

    # ── companyContacts
    for contact in c.get('companyContacts', []):
        if not isinstance(contact, dict):
            flag(c, f'companyContact is not dict: {contact!r}')
            continue
        ctype = contact.get('type', '')
        cval  = contact.get('value', '')
        if ctype not in VALID_CONTACT_TYPES:
            flag(c, f'unknown contact type: {ctype!r} = {cval!r}')
        if not cval or not cval.strip():
            flag(c, f'empty contact value for type {ctype!r}')
        if ctype == 'email' and not EMAIL_RE.match(cval):
            flag(c, f'invalid email: {cval!r}')
        if ctype == 'website' and not WEBSITE_RE.match(cval):
            flag(c, f'website missing scheme: {cval!r}')

    # ── userContacts
    for uc in c.get('userContacts', []):
        if not isinstance(uc, dict):
            flag(c, f'userContact is not dict: {uc!r}')
        elif not uc.get('name') and not uc.get('phone'):
            flag(c, 'userContact with no name and no phone')

print(f'Companies with issues: {len(issues)} / {len(data)}\n')
for company, msgs in sorted(issues.items()):
    print(f'  {company}')
    for m in msgs:
        print(f'    → {m}')

Companies with issues: 0 / 3082



## Extended Type & Value Validation

In [33]:
import re
from urllib.parse import urlparse

FREE_DOMAINS = {"gmail.com","yahoo.com","yahoo.com.tw","hotmail.com","outlook.com",
                "icloud.com","qq.com","163.com","126.com","sina.com","foxmail.com"}

LABEL_KEYWORDS = re.compile(r"(?i)^(hotline|tel|fax|email|web|zalo|wechat|line|viber|skype)\s*:?$")
CJK_LABEL_RE   = re.compile(r"[\u4e00-\u9fff].*[：:]$")

PHONE_ARTIFACT_RE = re.compile(r"(?i)(fax|mst|email|web|zalo)\s*[:\：]")
IP_RE  = re.compile(r"^https?://\d{1,3}(\.\d{1,3}){3}")
TAXID_FAKE_RE = re.compile(r"^(\d)\1{9}$")  # all-same digit: 0000000000, 1111111111...

issues2 = {}
def flag2(c, msg):
    issues2.setdefault(name(c), []).append(msg)

for c in data:

    # ── Phone sanity ────────────────────────────────────────────────────────────
    for contact in c.get("companyContacts", []):
        if contact["type"] not in ("tel", "fax", "hotline"):
            continue
        val = contact["value"]
        norm = re.sub(r"[\s\-\.\(\)]", "", val)

        if len(norm) < 8:
            flag2(c, f"phone too short ({len(norm)} digits): {val!r}")
        elif len(norm) > 15:
            flag2(c, f"phone too long ({len(norm)} chars) — likely parsing artifact: {val!r}")

        if PHONE_ARTIFACT_RE.search(val):
            flag2(c, f"phone contains embedded label — parsing artifact: {val!r}")

    # ── Contact values with embedded labels (parsing artifacts) ─────────────────
    CONTACT_EMBED_RE = re.compile(
        r'\b(Tel|Fax|Email|Web|Website|Zalo|Wechat|Line|Viber|Facebook|Skype|MST|Hotline)\s*[:\：]',
        re.IGNORECASE
    )
    for contact in c.get("companyContacts", []):
        if contact["type"] in ("tel", "fax", "hotline"):
            continue  # already checked by PHONE_ARTIFACT_RE above
        val = contact["value"]
        if CONTACT_EMBED_RE.search(val):
            flag2(c, f"contact value has embedded label [{contact['type']}]: {val!r}")

    # ── Website sanity ──────────────────────────────────────────────────────────
    for contact in c.get("companyContacts", []):
        if contact["type"] != "website":
            continue
        val = contact["value"]
        if IP_RE.match(val):
            flag2(c, f"website is IP address: {val!r}")
        try:
            parsed = urlparse(val)
            host = parsed.netloc or parsed.path
            if "." not in host:
                flag2(c, f"website has no TLD: {val!r}")
        except Exception:
            flag2(c, f"website unparseable: {val!r}")




    # ── userContact names that look like labels ──────────────────────────────────
    for uc in c.get("userContacts", []):
        n = uc.get("name", "") or ""
        if LABEL_KEYWORDS.match(n.strip()) or CJK_LABEL_RE.match(n.strip()):
            flag2(c, f"userContact name looks like a label: {n!r}")

    # ── Tax ID obviously fake ────────────────────────────────────────────────────
    tid = c.get("taxId") or ""
    if TAXID_FAKE_RE.match(tid):
        flag2(c, f"tax ID looks fake (all same digit): {tid!r}")


print(f"Companies with extended issues: {len(issues2)} / {len(data)}\n")
SEP2 = "-" * 50
CATEGORIES = {
    "phone": "PHONE",
    "contact value has embedded": "CONTACT EMBED ARTIFACT",
    "website": "WEBSITE",
    "userContact name": "LABEL AS USER NAME",
    "tax ID": "FAKE TAX ID",
}
grouped: dict[str, list[tuple[str,str]]] = {}
for company, msgs in issues2.items():
    for msg in msgs:
        cat = next((v for k, v in CATEGORIES.items() if k in msg), "OTHER")
        grouped.setdefault(cat, []).append((company, msg))

for cat, entries in sorted(grouped.items()):
    print(f"\n{cat} ({len(entries)})")
    print(SEP2)
    for company, msg in entries[:20]:
        print(f"  {company}")
        print(f"    → {msg}")
    if len(entries) > 20:
        print(f"  ... and {len(entries)-20} more")


Companies with extended issues: 0 / 3082



### 5. Email ↔ Website Type Mismatch
Contacts where an email is stored as  type, or a URL is stored as  type.
Usually caused by regex misclassification or unhandled separator (space / newline).

In [34]:
import re

EMAIL_PATTERN = re.compile(r"^[^\@\s]+@[^\@\s]+\.[^\@\s]+$")
URL_PATTERN   = re.compile(r"^https?://")

cross_type = []  # (company_name, contact_type, value, issue)

for c in data:
    for contact in c.get("companyContacts", []):
        ctype = contact["type"]
        val   = contact["value"].strip()

        if ctype == "website":
            if EMAIL_PATTERN.match(val):
                cross_type.append((name(c), ctype, val, "email stored as website"))
            elif "@" in val and not URL_PATTERN.match(val):
                cross_type.append((name(c), ctype, val, "website contains @ but no scheme"))

        elif ctype == "email":
            if URL_PATTERN.match(val):
                cross_type.append((name(c), ctype, val, "URL stored as email"))
            elif " " in val or "\n" in val:
                cross_type.append((name(c), ctype, val, "email has whitespace — possible concatenation"))
print(f"Cross-type mismatches: {len(cross_type)}\n")
by_issue: dict = {}
for company, ctype, val, issue in cross_type:
    by_issue.setdefault(issue, []).append((company, ctype, val))

SEP = "-" * 50
for issue, entries in sorted(by_issue.items()):
    print(f"\n{issue.upper()} ({len(entries)})")
    print(SEP)
    for company, ctype, val in entries[:20]:
        print(f"  [{ctype}] {company}")
        print(f"    -> {val!r}")
    if len(entries) > 20:
        print(f"  ... and {len(entries)-20} more")


Cross-type mismatches: 0

